# PROBLEM: Teach a robot to identify fruit

## ALGORITHM: DecisionTreeClassifier

In [1]:
!pip install mlflow

In [2]:
import pandas as pd
from pyspark.sql import SparkSession

import mlflow
import mlflow.spark

from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, Imputer, StandardScaler
from pyspark.ml.pipeline import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder


### STEP 1: LOAD CSV & START SPARK
This is like waking up the robot and giving it its textbook.

In [3]:
data = {
    "weight": [150, 170, 50, 60, 160, 55, None], # Some data is missing (None)!
    "color": ["Red", "Red", "Yellow", "Yellow", "Red", "Yellow", "Red"],
    "label": ["Apple", "Apple", "Banana", "Banana", "Apple", "Banana", "Apple"]
}

# Save it to a CSV file so the robot can read it later
pd.DataFrame(data).to_csv("work/dataset/fruits.csv", index=False)

In [4]:
spark = SparkSession.builder.appName("FruitClassification").getOrCreate()
df = spark.read.csv("work/dataset/fruits.csv", header=True, inferSchema=True)

### STEP 2: SETUP MLFLOW TRACKING
This opens the "Lab Notebook" to record our experiment.

In [5]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.pyspark.ml.autolog() 
mlflow.set_experiment("Fruit_Classification")

with mlflow.start_run(run_name="DecisionTreeModel"):

    ### STEP 3: FEATURE ENGINEERING (IMPUTER & INDEXERS)
    ### We fix missing data and translate words into "Robot Language" (numbers)
    imputer = Imputer(inputCols=['weight'],outputCols=['weight_imputed'])
    # peek into the imputer's magic box
    imputer_model = imputer.fit(df)
    imputed_df = imputer_model.transform(df)
    imputed_df.select("weight", "weight_imputed").show()

    color_indexer = StringIndexer(inputCol="color", outputCol="color_indexed")
    color_model = color_indexer.fit(df)
    #  Ask the robot: "What does 0 mean? What does 1 mean?"
    print(color_model.labels)
    color_indexed_df = color_model.transform(df)
    color_indexed_df.select("color", "color_indexed").show()
    
    label_indexer = StringIndexer(inputCol="label", outputCol="label_indexed")
    label_model = label_indexer.fit(df)
    print(label_model.labels)
    label_indexed_df = label_model.transform(df)
    label_indexed_df.select("label", "label_indexed").show()


    ### STEP 4: PREPARE FEATURES (ENCODER & ASSEMBLER)
    ### We organize the numbers into a neat "Feature Vector" (a digital suitcase).
    encoder = OneHotEncoder(inputCols=["color_indexed"], outputCols=["color_vec"])
    assembler = VectorAssembler(inputCols=["weight_imputed", "color_vec"], outputCol="features_unscaled")

    ### STEP 5: THE LEVELER (STANDARD SCALER)
    ### This makes sure the "weight" and "color" numbers play fair together.
    scaler = StandardScaler(inputCol="features_unscaled", outputCol="features") 

    ### STEP 6: DEFINE THE BRAIN (DECISION TREE)
    ### Note: We tell the brain to look at the "features" (the scaled ones!)
    dt = DecisionTreeClassifier(labelCol="label_indexed", featuresCol="features")

    ### STEP 7: BUILD THE ASSEMBLY LINE (PIPELINE)
    ### We added the 'scaler' right before the 'dt' (the brain).
    pipeline = Pipeline(stages=[imputer, color_indexer, label_indexer, encoder, assembler, scaler, dt])


    ### STEP 8: HYPERPARAMETER TUNING
    grid = ParamGridBuilder().addGrid(dt.maxDepth, [2, 5]).build()
    tvs = TrainValidationSplit(estimator=pipeline, 
                                 estimatorParamMaps=grid, 
                                 evaluator=MulticlassClassificationEvaluator(labelCol="label_indexed"))

    ### STEP 9: TRAIN & LOG
    model = tvs.fit(df)
    mlflow.spark.log_model(model.bestModel, "scaled_fruit_robot")
    
    print("Robot is trained and the numbers are perfectly balanced!")
    accuracy = MulticlassClassificationEvaluator(labelCol="label_indexed").evaluate(model.transform(df))
    #mlflow.log_metric("accuracy", accuracy)
    #mlflow.log_param("best_maxDepth", model.bestModel.stages[-1].getMaxDepth())
    mlflow.spark.log_model(model.bestModel, "best_fruit_robot")


MlflowException: API request to http://localhost:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='localhost', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Fruit_Classification (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7936acb72390>: Failed to establish a new connection: [Errno 111] Connection refused'))

### STEP 3: RUN THE MLFLOW

In Powershell
- docker exec -it spark_dev bash

To run mlflow
- pip install mlflow -q
- mlflow server --host 0.0.0.0 --port 5000 --backend-store-uri /home/jovyan/work/mlruns

Access
- http://127.0.0.1:5000